In [1]:

import os
import shutil


from google.colab import userdata
USER = "Koubra-gaby"
#TOKEN = userdata.get('GH_TOKEN')
REPO_NAME = "Chadian_arabic-update"

if os.path.exists(REPO_NAME):
    print(f"Le dossier '{REPO_NAME}' existe déjà. Suppression en cours...")
    shutil.rmtree(REPO_NAME)
# Cloner le dépôt privé
!git clone https://{USER}:@github.com/{USER}/{REPO_NAME}.git


Cloning into 'Chadian_arabic-update'...
remote: Enumerating objects: 499, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 499 (delta 88), reused 49 (delta 21), pack-reused 348 (from 1)
Receiving objects: 100% (499/499), 24.37 MiB | 18.82 MiB/s, done.
Resolving deltas: 100% (252/252), done.


In [2]:
# via python & Jupyter
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('update_shu'))

## training fastext model for chadian arabic corpus

In [8]:
%cd /content/Chadian_arabic-update

/content/Chadian_arabic-update


In [36]:
with open("/content/Chadian_arabic-update/Data/french_nllb_espace.vec", encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline().strip())

23685 1024
A -0.03778076 -0.00822449 0.01852417 0.00742340 0.01747131 -0.02383423 0.02618408 -0.02221680 -0.01890564 -0.07098389 -0.02369690 -0.02026367 -0.03967285 -0.00908661 -0.05880737 0.01853943 0.08966064 0.04403687 0.07244873 0.03005981 0.12951660 -0.07727051 0.01184082 -0.02563477 0.03010559 0.24182129 -0.20861816 0.15368652 0.01383209 -0.12988281 -0.08093262 -0.05004883 -0.08410645 -0.02044678 -0.02886963 0.13549805 -0.10888672 0.03683472 -0.12634277 -0.11822510 -0.04492188 -0.07965088 -0.32446289 0.04174805 -0.12634277 0.39697266 -0.02671814 -0.18444824 0.14721680 -0.27197266 0.18493652 0.10540771 0.10961914 -0.06970215 0.06866455 -0.02821350 0.08215332 -0.18420410 0.16174316 0.14953613 -0.06787109 0.08331299 -0.01028442 -0.06872559 0.16503906 -0.17907715 0.09240723 0.04901123 -0.07373047 0.01476288 -0.02258301 -0.24829102 0.19970703 -0.14611816 0.06134033 0.23059082 0.04528809 -0.02680969 -0.03759766 -0.09393311 0.02296448 -0.03099060 0.12481689 -0.00337601 -0.23901367 0.131

In [37]:
from gensim.models import KeyedVectors

vectors = KeyedVectors.load_word2vec_format(
    "/content/Chadian_arabic-update/Data/french_nllb_espace.vec",
    binary=False
)

print("Vocabulaire :", len(vectors))
print("Dimension :", vectors.vector_size)

Vocabulaire : 23685
Dimension : 1024


In [43]:
word = "salut"

similar_words = vectors.most_similar(
    word,
    topn=10
)

for similar_word, score in similar_words:
    print(f"{similar_word:<20} {score:.4f}")

salutation           0.7211
salutations          0.7086
Salut                0.5985
salutaire            0.2772
Salu                 0.2369
saluerez             0.2360
saluer               0.2341
saluent              0.2265
salués               0.2233
salué                0.2228


In [44]:
!git clone https://github.com/facebookresearch/MUSE.git

Cloning into 'MUSE'...
remote: Enumerating objects: 239, done.
remote: Total 239 (delta 0), reused 0 (delta 0), pack-reused 239 (from 1)
Receiving objects: 100% (239/239), 215.77 KiB | 35.96 MiB/s, done.
Resolving deltas: 100% (136/136), done.


In [46]:
%cd /content/MUSE

from pathlib import Path

path = Path("src/utils.py")
code = path.read_text()

code = code.replace(
    "inspect.getargspec(optim_fn.__init__)[0]",
    "inspect.getfullargspec(optim_fn.__init__).args"
)

path.write_text(code)

print("Modification appliquée.")

/content/MUSE
Modification appliquée.


In [47]:
file_path = "/content/MUSE/src/trainer.py"

with open(file_path, "r") as f:
    content = f.read()

content = content.replace(
    "torch.load(path)",
    "torch.load(path, weights_only=False)"
)

with open(file_path, "w") as f:
    f.write(content)

print("Correction appliquée.")

Correction appliquée.


In [49]:
!pip install -U faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 79.3 MB/s  0:00:01


In [50]:
!python unsupervised.py \
    --src_lang shu \
    --tgt_lang fr \
    --src_emb /content/Chadian_arabic-update/Data/arabic_embedding.vec \
    --tgt_emb /content/Chadian_arabic-update/Data/french_nllb_espace.vec \
    --emb_dim 1024 \
    --n_refinement 10 \
    --n_epochs 10 \
    --dis_most_frequent 20000 \
    --dico_eval /content/Chadian_arabic-update/Data/dico_eval.txt

INFO - 11/05/26 17:50:38 - 0:00:00 - ============ Initialized logger ============
INFO - 11/05/26 17:50:38 - 0:00:00 - adversarial: True
                                     batch_size: 32
                                     cuda: True
                                     dico_build: S2T
                                     dico_eval: /content/Chadian_arabic-update/Data/dico_eval.txt
                                     dico_max_rank: 15000
                                     dico_max_size: 0
                                     dico_method: csls_knn_10
                                     dico_min_size: 0
                                     dico_threshold: 0
                                     dis_clip_weights: 0
                                     dis_dropout: 0.0
                                     dis_hid_dim: 2048
                                     dis_input_dropout: 0.1
                                     dis_lambda: 1
                                     dis_layers: 2
 

# Tester le modele entrainer

In [51]:
import numpy as np

## Get nearest neighbors

In [54]:
src_path = '/content/MUSE/dumped/debug/dtxwjv76ku/vectors-shu.txt'
tgt_path = '/content/MUSE/dumped/debug/dtxwjv76ku/vectors-fr.txt'
nmax = 50000  # maximum number of word embeddings to load

src_embeddings, src_id2word, src_word2id = load_vec(src_path, nmax)
tgt_embeddings, tgt_id2word, tgt_word2id = load_vec(tgt_path, nmax)

In [58]:
# printing nearest neighbors in the source space
src_word = 'ana'
get_nn(src_word, src_embeddings, src_id2word, src_embeddings, src_id2word, K=5)

Nearest neighbors of "ana":
1.0000 - ana
0.8043 - nisiitni
0.7972 - Ana
0.7945 - nafsiana
0.7927 - zaati


In [69]:
# printing nearest neighbors in the target space
src_word = 'moi'
get_nn(src_word, tgt_embeddings, tgt_id2word, src_embeddings, src_id2word, K=5)

Nearest neighbors of "moi":
0.1085 - hayaatha
0.1040 - hayne
0.1037 - khadamtaha
0.1026 - nukhuttaha
0.1026 - badalha


# Projection vectorielle

In [ ]:
%%capture
!pip install -q gensim transformers scipy sentencepiece

### Sauvegarder le modèle, le tokenizer et la transformation

In [78]:
%cd /content/Chadian_arabic-update

/content/Chadian_arabic-update


In [87]:
!git reset --soft HEAD~2


In [88]:
!git lfs install


Updated Git hooks.
Git LFS initialized.


In [89]:
!git lfs track "*.txt"


"*.txt" already supported


In [90]:
!git add /content/Chadian_arabic-update/Data/vectors-fr.txt /content/Chadian_arabic-update/Data/vectors-shu.txt

In [91]:
!git commit -m "update"

[main 6ceeedf] update
 2 files changed, 6 insertions(+)
 create mode 100644 Data/vectors-fr.txt
 create mode 100644 Data/vectors-shu.txt


In [92]:
!git push

Uploading LFS objects: 100% (2/2), 526 MB | 0 B/s, done.
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 599 bytes | 599.00 KiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
remote: This repository moved. Please use the new location:
remote:   https://github.com/koubra-gaby/Chadian_arabic-update.git
To https://github.com/Koubra-gaby/Chadian_arabic-update.git
   72c0744..6ceeedf  main -> main
